# MuJoCo Inverse Dynamics: Built-in Functions
- use "mujoco.mj_inverse"

#### 1. Load scene

In [1]:
import os
import sys
import numpy as np
import time

import mujoco

sys.path.append(os.path.abspath('../'))
from pp_base_mujoco.VIEWER import *
from pp_base_mujoco.UTILS import *

In [2]:
xml_path = '../asset/panda_scene.xml'
# xml_path = '../asset/ur_scene.xml'
xml_abs_path = os.path.abspath(xml_path)

model = mujoco.MjModel.from_xml_path(xml_abs_path)
data = mujoco.MjData(model)

#### 2. Ingredients for Inverse Dynamics
- qfrc
- Mass, Coriolis, Gravity

In [3]:
qpos_init = np.array([0, -0.5, 0, -2.5, 0, 2.0, 0.5]) # set initial qpos
data.qpos[:] = qpos_init
mujoco.mj_forward(model, data) # calculate initial dynamics

In [7]:
""" CALCULATE MASS MATRIX """
M_full = np.zeros((model.nv, model.nv), dtype=np.float64)
mujoco.mj_fullM(model, M_full, data.qM)
print("Mass matrix: ", M_full)

""" BIAS FORCE: CORIOLIS AND GRAVITY """
print("Bias force: ", data.qfrc_bias)

Mass matrix:  [[ 1.83417863e+00  3.39739386e-16  1.18362335e+00  6.13893015e-16
  -1.61204314e-01  2.66973675e-16 -1.00000000e-01]
 [ 3.39739386e-16  2.17964996e+00  3.37929891e-16 -1.05148375e+00
  -4.19872098e-17 -2.54557152e-01 -2.39848364e-17]
 [ 1.18362335e+00  3.37929891e-16  1.45919025e+00  5.72962079e-16
  -3.16883799e-01  2.35449471e-16 -8.77582562e-02]
 [ 6.13893015e-16 -1.05148375e+00  5.72962079e-16  1.29133417e+00
  -8.64513096e-17  2.98184586e-01 -5.37307754e-17]
 [-1.61204314e-01 -4.19872098e-17 -3.16883799e-01 -8.64513096e-17
   4.03400772e-01 -8.10739782e-17  4.16146837e-02]
 [ 2.66973675e-16 -2.54557152e-01  2.35449471e-16  2.98184586e-01
  -8.10739782e-17  2.22015125e-01 -9.81396964e-17]
 [-1.00000000e-01 -2.39848364e-17 -8.77582562e-02 -5.37307754e-17
   4.16146837e-02 -9.81396964e-17  1.00000000e-01]]
Bias force:  [ 0.00000000e+00 -8.95045641e+00 -1.28348632e-15  1.72184407e+01
  2.29146266e-17  1.74618000e+00 -1.23259516e-32]


In [10]:
""" EXTERNAL FORCE """
print("external force applied: ", data.qfrc_applied)

""" TORQUE CONTTROL  """
print("Torque control: ", data.ctrl)
print("torque actually applied: ", data.qfrc_actuator)

external force applied:  [0. 0. 0. 0. 0. 0. 0.]
Torque control:  [0. 0. 0. 0. 0. 0. 0.]
torque actually applied:  [0. 0. 0. 0. 0. 0. 0.]


In [11]:
""" INVERSE DYNAMICS """
mujoco.mj_inverse(model, data) # call inverse dynamics 
print("torque calculated by inverse dynamics: ", data.qfrc_inverse)

torque calculated by inverse dynamics:  [ 1.59360675e-30 -1.77635684e-15  2.36658272e-30  7.10542736e-15
  1.12166160e-30 -4.44089210e-16  4.88279952e-32]


In [15]:
""" COMPARE INVERSE DYNAMICS AND MANUALLY CALCULATED TORQUE """
# 1. desired acceleration
desired_qacc = np.zeros(model.nv)
data.qacc[:] = desired_qacc
mujoco.mj_forward(model, data) # update dynamics with desired acceleration

# 2. calculate torque 
torque_calculated = M_full @ data.qacc + data.qfrc_bias 
print("torque calculated by mass matrix and bias force: ", torque_calculated)
torque_mujoco = torque_calculated - data.qfrc_passive - data.qfrc_constraint
print("actual torque with passive torque: ", torque_mujoco)
print("torque calculated by inverse dynamics: ", data.qfrc_inverse)

torque calculated by mass matrix and bias force:  [ 3.15544362e-30 -1.77635684e-15  1.38050658e-30  3.55271368e-15
  1.13398755e-30 -4.44089210e-16 -1.23259516e-32]
actual torque with passive torque:  [ 3.15544362e-30 -1.77635684e-15  1.38050658e-30  3.55271368e-15
  1.13398755e-30 -4.44089210e-16 -1.23259516e-32]
torque calculated by inverse dynamics:  [ 1.59360675e-30 -1.77635684e-15  2.36658272e-30  7.10542736e-15
  1.12166160e-30 -4.44089210e-16  4.88279952e-32]


#### 3-1. Compensation

In [ ]:
# 1. calculate with mujoco inverse 
torque_command = data.qfrc_inverse

""" MAIN LOOP """
viewer = MUJOCOGLVIEWER(model, data)
mujoco.mj_resetData(model, data)
data.qpos[:] = qpos_init
mujoco.mj_forward(model, data)

while viewer.is_alive():
    # calculate desired torque 
    data.qacc[:] = np.zeros(model.nv) # desired acceleration
    mujoco.mj_inverse(model, data) 
    torque_command = data.qfrc_inverse
    # apply control 
    data.ctrl[:] = torque_command
    mujoco.mj_step(model, data)
    viewer.render()

viewer.close()
del(viewer)

In [ ]:
# 2. manually calculate  
torque_calculated = data.qfrc_bias + M_full @ data.qacc
torque_actual = data.qfrc_bias + M_full @ data.qacc - data.qfrc_passive - data.qfrc_constraint

""" MAIN LOOP """
viewer = MUJOCOGLVIEWER(model, data)
mujoco.mj_resetData(model, data)
data.qpos[:] = qpos_init
mujoco.mj_forward(model, data)

while viewer.is_alive():
    # calculate desired torque 
    data.qacc[:] = np.zeros(model.nv) # desired acceleration
    M_full = np.zeros((model.nv, model.nv), dtype=np.float64)
    mujoco.mj_fullM(model, M_full, data.qM)
    torque_command = data.qfrc_bias + M_full @ data.qacc # static state
    # apply control 
    data.ctrl[:] = torque_command
    mujoco.mj_step(model, data)
    viewer.render()

viewer.close()
del(viewer)